
# 1.  Why Pydantic Exists?
Python won't stop you from shipping garbage data. Pydantic is the fix — but first, the foundation it's built on: type hints.

### The dynamic typing problem
Python is dynamically typed. A single variable can hold an integer, then a string, then a list, with zero complaints from the language itself. This flexibility is genuinely useful for quick scripts — and genuinely dangerous the moment you're working with data that came from *outside* your own code: an API response, a form submission, a file upload, or an LLM.

Consider a typical register_user() function that receives a dictionary and calculates a birth year from an age field. It looks completely reasonable. It works perfectly — right up until age arrives as the string "unknown" instead of a number, at which point the program crashes deep inside a calculation, often several function calls away from where the bad data actually entered the system.

The bug was never really in the calculation. The bug was that nothing checked the incoming data at the door.


In [1]:
def register_user(name, email, age):
    birth_year = 2026 - age   # assumes age is an int — nothing enforces that
    print(f"Registered {name}, born approx. {birth_year}")

# Looks fine...
register_user("Aditi", "aditi@example.com", 28)

# ...until real-world data shows up like this:
register_user("Rohan", "rohan@example.com", "unknown")
# TypeError: unsupported operand type(s) for -: 'int' and 'str'
# Notice WHERE it crashed: deep inside the function, not at the door.

Registered Aditi, born approx. 1998


TypeError: unsupported operand type(s) for -: 'int' and 'str'

### Type hints: documentation, not enforcement
Type hints are the foundation everything else in this guide is built on. They tell Python — and every developer reading the code — what type a variable is *supposed* to hold: `name: str`, `age: int`, `price: float`, `is_active: bool`.

Here's the part that trips people up: **Python does not enforce type hints at runtime.** This line runs without a single complaint: `age: int = "not a number at all"`. Type hints are read by humans, by your IDE (for autocomplete and error squiggles), and — critically — by Pydantic. But plain Python itself ignores them completely.

Container types follow the same pattern: `list[str]` for a list of strings, `dict[str, int]` for a dictionary mapping strings to integers. Since Python 3.9, use these lowercase built-ins directly rather than importing `List` / `Dict` from `typing` — you'll still see the older style in existing codebases, but the lowercase form is the modern standard.

In [2]:
# The four basic types you'll use constantly
name: str = "Aditi"
age: int = 28
price: float = 499.99
is_active: bool = True

# Container types
tags: list[str] = ["python", "pydantic", "fastapi"]
word_counts: dict[str, int] = {"error": 12, "warning": 5}

# Function signatures — the most valuable place to use hints
def format_price(amount: float, currency: str = "USD") -> str:
    return f"{currency} {amount:.2f}"

# The gotcha: Python enforces NONE of this
age: int = "not a number at all"   # runs. no error. no warning.

### Optional and Literal types
Two more type-hint patterns show up constantly once Pydantic enters the picture — and become especially important later when constraining what an AI model is allowed to return.

`Optional` (or the modern `| None` syntax) marks a value that might legitimately not exist yet: `phone: str | None = None`. Both `Optional[str]` and `str | None` mean exactly the same thing; the pipe syntax is preferred in modern Python (3.10+).

`Literal` restricts a value to an exact, specific set of options — a multiple-choice question instead of a fill-in-the-blank: `status: Literal["draft", "published", "archived"]`. On its own (still without Pydantic), this is only documentation. Once Pydantic enforces it, `Literal` becomes a genuinely powerful tool — especially for constraining AI-generated classifications, covered in depth later.
 

In [ ]:
from typing import Optional, Literal

# Optional — old style and modern style mean the same thing
middle_name: Optional[str] = None
phone: str | None = None  # means it can either contain string or it will contain none

# Literal — a strict multiple-choice constraint
status: Literal["draft", "published", "archived"] = "draft"
priority: Literal["low", "medium", "high"] = "medium"

# Still just documentation at this stage — Python allows this:
status = "this is not one of the allowed options"  # no error yet

# 2. Your First Model
BaseModel vs. dataclass vs. a plain class — and why only one of them actually validates anything.
### BaseModel vs. dataclass vs. plain class
This is the moment everything from Part 1 comes together. Three ways exist to define "the shape of some data" in Python — a plain class, a `@dataclass`, and a Pydantic `BaseModel`. They look almost identical. They behave very differently.

A plain class with only type-hinted attributes and no `__init__` doesn't even accept constructor arguments — it fails immediately, but for a confusing reason unrelated to data validation.

A `@dataclass` generates a real `__init__` from the type hints, so it accepts arguments — but validates absolutely nothing. Passing `age="not a number"` is accepted silently.

A `BaseModel` looks the same on the surface, but genuinely inspects the incoming data: it validates types, coerces safely-compatible values, and raises a clear `ValidationError` the moment something doesn't fit — before the object is even created.

In [ ]:
from dataclasses import dataclass
from pydantic import BaseModel

# Option 1: dataclass — clean syntax, ZERO validation
@dataclass
class UserDataclass:
    name: str
    email: str
    age: int

user = UserDataclass(name="Alice", email="alice@example.com", age="not a number")
print(user.age)   # "not a number" — accepted with no complaint at all

# Option 2: BaseModel — actually inspects the data
class UserModel(BaseModel):
    name: str
    email: str
    age: int

UserModel(name="Alice", email="alice@example.com", age="not a number")
# ValidationError: Input should be a valid integer,
# unable to parse string as an integer

# But a numeric STRING is coerced safely:
user = UserModel(name="Alice", email="alice@example.com", age="30")
print(user.age, type(user.age))   # 30 <class 'int'>

not a number
30 <class 'int'>


### Creating instances, coercion, required vs. optional
There are two equivalent ways to create a model instance from a dictionary: unpacking with `Model(**data)`, or calling `Model.model_validate(data)`. Both validate identically; `model_validate()` is the better choice when you need extra options later, such as `strict=True`.

Fields without a default value are required — Pydantic raises a `ValidationError` if they're missing, and the error names every problem field at once, not just the first one it finds. Fields with a default (like `newsletter_opt_in: bool = False`) are optional.

The structured format of these validation errors matters far beyond debugging convenience — it's exactly what powers the "ask an AI model to retry with the error fed back to it" pattern covered in the AI Bridge section later in this guide.

In [4]:
from pydantic import BaseModel, ValidationError

class SignupForm(BaseModel):
    username: str
    email: str
    age: int
    newsletter_opt_in: bool = False   # has a default -> optional

# Two equivalent creation styles
user_a = SignupForm(**{"username": "aditi28", "email": "a@x.com", "age": 28})
user_b = SignupForm.model_validate({"username": "aditi28", "email": "a@x.com", "age": 28})

try:
    SignupForm(username="incomplete_user")   # missing email AND age
except ValidationError as e:
    print(e)
# 2 validation errors for SignupForm
# email
#   Field required [type=missing, ...]
# age
#   Field required [type=missing, ...]

2 validation errors for SignupForm
email
  Field required [type=missing, input_value={'username': 'incomplete_user'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
age
  Field required [type=missing, input_value={'username': 'incomplete_user'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


### Serialization basics: model_dump() and model_dump_json()
Once data is validated *into* a model, `model_dump()` and `model_dump_json()` get it back *out* in a shape the rest of your system can use.

`model_dump()` returns a plain Python dictionary — use it when inserting into a database, or passing to another function expecting a dict. `model_dump_json()` returns a JSON string directly — use it for HTTP API responses or writing to a `.json` file. Both accept `indent=` for pretty-printed, human-readable output.

In [9]:
user = SignupForm(username="aditi28", email="aditi@example.com", age=28, newsletter_opt_in=True)

user.model_dump()
# {'username': 'aditi28', 'email': 'aditi@example.com', 'age': 28, 'newsletter_opt_in': True}

user.model_dump_json()
# '{"username":"aditi28","email":"aditi@example.com","age":28,"newsletter_opt_in":true}'

user.model_dump_json(indent=2)   # pretty-printed for humans/logs

'{\n  "username": "aditi28",\n  "email": "aditi@example.com",\n  "age": 28,\n  "newsletter_opt_in": true\n}'

# 3. Field Constraints & Custom Validators
Types alone aren't enough. Enforcing real business rules with Field(), field_validator, and model_validator.

### Field() constraints and the Annotated pattern
`age: int` accepts `-50` just as happily as `28`. Real rules — minimum lengths, numeric ranges, regex patterns — need `Field()`.

Numeric constraints: `gt` (greater than), `ge` (greater than or equal to), `lt` (less than), `le` (less than or equal to). String constraints: `min_length`, `max_length`, `pattern` (a regex the value must match).

Two equivalent calling styles exist. The direct style — `age: int = Field(gt=0, le=120)` — is simple and common. The `Annotated` style — `age: Annotated[int, Field(gt=0, le=120)]` — separates "the type" from "the metadata about the type," which becomes valuable once you stack multiple pieces of metadata (constraints, descriptions, examples) on one field. Prefer `Annotated` in production code; both behave identically.

In [11]:
from typing import Annotated
from pydantic import BaseModel, Field

# Direct style
class JobApplicationV1(BaseModel):
    full_name: str = Field(min_length=2, max_length=100)
    years_experience: int = Field(ge=0, le=50)
    portfolio_url: str = Field(pattern=r"^https?://.*")

# Annotated style — same behavior, more composable
class JobApplicationV2(BaseModel):
    full_name: Annotated[str, Field(min_length=2, max_length=100)]
    years_experience: Annotated[int, Field(ge=0, le=50)]
    email: Annotated[
        str,
        Field(description="Applicant's contact email", examples=["rohan@example.com"]),
    ]

### Built-in special types

Pydantic ships pre-built types for extremely common validation needs, so common patterns don't need to be hand-rolled as regexes. `EmailStr` validates real email format (requires the `email` extra: `pip install pydantic[email]`). `HttpUrl` / `AnyUrl` validate URL format. `SecretStr` wraps sensitive values so they never print or log in plaintext by default — access the real value on purpose with `.get_secret_value()`.

In [13]:
from pydantic import BaseModel, EmailStr, HttpUrl, SecretStr

class Applicant(BaseModel):
    name: str
    email: EmailStr        # rejects "not-an-email"
    website: HttpUrl        # must be http:// or https://

class UserAccount(BaseModel):
    username: str
    password: SecretStr

account = UserAccount(username="rohan99", password="super-secret-123")
print(account)                              # password shows as **********
print(account.password.get_secret_value())  # the real value, on purpose

username='rohan99' password=SecretStr('**********')
super-secret-123


### field_validator: per-field custom logic

Built-in constraints cover most cases. For real business logic, write a validator function with `@field_validator`, which operates on exactly one field.

- `mode='after'` (the default) runs after Pydantic's normal type coercion — the value you receive is already the correct type. 
- `mode='before'` runs on the raw input before any type coercion is attempted, useful for cleaning up messy text (like `"5 years"`) before Pydantic tries to interpret it as an `int`.

Inside a validator, always either `return` the value (transformed or not) to accept it, or `raise ValueError(...)` to reject the whole model with a clear message.

In [14]:
from pydantic import BaseModel, field_validator

class JobApplication(BaseModel):
    full_name: str
    email: str
    years_experience: int

    @field_validator("full_name", mode="after")
    @classmethod
    def normalize_name(cls, value: str) -> str:
        cleaned = value.strip()
        if not cleaned:
            raise ValueError("full_name cannot be empty")
        return cleaned.title()

    @field_validator("years_experience", mode="before")
    @classmethod
    def strip_years_suffix(cls, value):
        # handles messy input like "5 years" arriving as raw text
        if isinstance(value, str):
            digits = "".join(ch for ch in value if ch.isdigit())
            return int(digits) if digits else value
        return value

app = JobApplication(full_name="  rohan mehta  ", email="r@x.com", years_experience="5 years")
print(app.full_name, app.years_experience)   # Rohan Mehta 5

Rohan Mehta 5


### `model_validator`: cross-field rules

Some rules can't be checked one field at a time — they depend on the relationship between *multiple* fields, like `"password must equal confirm_password."` A single `field_validator` has no visibility into any other field, so this is impossible to express there. `model_validator` receives the entire model, after every individual field has already passed its own checks.

Execution order matters and is fixed: every `field_validator` on a model runs first, for every field, before `model_validator` ever runs.

In [ ]:
from pydantic import BaseModel, model_validator

class SignupForm(BaseModel):
    username: str
    password: str
    confirm_password: str

    @model_validator(mode="after")
    def passwords_must_match(self):
        if self.password != self.confirm_password:
            raise ValueError("password and confirm_password do not match")
        return self

# A richer example — mutually exclusive preferences
class JobApplication(BaseModel):
    remote_preferred: bool
    willing_to_relocate: bool

    @model_validator(mode="after")
    def check_relocation_logic(self):
        if self.remote_preferred and self.willing_to_relocate:
            raise ValueError("Can't be both remote-only AND willing to relocate")
        return self

# 04. Computed Fields & Serialization Control

Deriving values instead of storing stale ones, and precisely controlling what leaves a model.

### `@computed_field`

Some values shouldn't be stored at all — they should be derived, live, every time, from other fields. `@computed_field` (stacked with `@property`) turns a normal Python property into something that also appears automatically in `model_dump()` and the generated JSON schema.

It becomes a read-only attribute: accessible like a normal field, but never settable through the constructor — there's no `experience_tier=` argument, and there never should be, because it's always derived.

In [16]:
from pydantic import BaseModel, computed_field

class JobApplication(BaseModel):
    full_name: str
    years_experience : int

    @computed_field
    @property
    def experience_tier(self) -> str:
        if self.years_experience < 2:
            return "Junior"
        elif self.years_experience < 7:
            return "Mid"
        return "senior"

app = JobApplication(full_name="Aditi Sharma", years_experience=6)
print(app.experience_tier)      # "mid" — accessed like a normal attribute
print(app.model_dump())         # includes experience_tier automatically

app.years_experience = 10       # models are mutable by default
print(app.experience_tier)      # "senior" — always fresh, never stale

Mid
{'full_name': 'Aditi Sharma', 'years_experience': 6, 'experience_tier': 'Mid'}
senior


## Serialization control: `exclude`, `include`, `exclude_unset`

`model_dump()` returns every field by default. Real applications almost always need to hide something (a password) or send only part of a model.

`exclude={"password"}` removes specific fields from the output. `include={"username"}` is the inverse — only those fields, nothing else. `exclude_unset=True` is the single most useful flag for PATCH-style partial updates: it only serializes fields the caller explicitly provided, meaning default values (like an unset `bio` field) disappear from the output rather than silently overwriting other data. `exclude_none=True` drops any field currently set to `None`.

In [17]:
user = UserAccount(username="rohan99", email="rohan@example.com", password="hunter2")

user.model_dump(exclude={"password"})
# {'username': 'rohan99', 'email': 'rohan@example.com'}

user.model_dump(include={"username"})
# {'username': 'rohan99'}

# The PATCH-update use case:
user.model_dump(exclude_unset=True)
# Only fields the caller actually SET appear — defaults that were
# never explicitly provided are omitted entirely.

{'username': 'rohan99', 'password': SecretStr('**********')}

# 05. Nested Models

Real data is almost never flat. One model as another's field type, and validation cascades automatically.

### Models inside models, lists of models

A job application has an applicant, who has an address. An order has a customer and a list of items. Using one `BaseModel` as a field type inside another just works — and validation cascades automatically through every layer.

The real power shows when parsing directly from a nested dictionary — exactly what happens when JSON arrives from an API, a form, or (later in this guide) an LLM response. Pydantic builds the nested model automatically from the inner dictionary, and dot-chain access works exactly like any normal nested Python object.

When something goes wrong deep inside a nested structure, the validation error pinpoints the exact nested path — `address.state`, `address.pin_code` — no matter how many layers deep the problem is buried.

In [18]:
from pydantic import BaseModel

class Address(BaseModel):
    city: str
    state: str
    pin_code: str

class Applicant(BaseModel):
    name: str
    email: str
    address: Address        # a whole model, used as a field type

# Parsing straight from a nested dictionary — the common real-world case
incoming = {
    "name": "Rohan Mehta",
    "email": "rohan@example.com",
    "address": {"city": "Pune", "state": "Maharashtra", "pin_code": "411001"},
}
applicant = Applicant.model_validate(incoming)
print(applicant.address.city)   # "Pune" — dot-chain access, fully typed

# Lists of nested models work the same way
class WorkExperience(BaseModel):
    company: str
    role: str
    years: int

class Application(BaseModel):
    applicant: Applicant
    work_history: list[WorkExperience]

Pune


### Optional nested models and deep nesting

An entire nested object can be made optional with `SomeModel | None = None` — not just its individual fields, but the whole object may simply be absent, which is common for things like a shipping address that doesn't exist until checkout is complete.

Nesting has no meaningful depth limit. Model inside model inside list of models works exactly as expected, with full IDE autocomplete at every level in any real editor.

In [ ]:
class Discount(BaseModel):
    code: str
    percent_off: float

class Order(BaseModel):
    order_id: str
    total: float
    discount: Discount | None = None   # the WHOLE nested object may be absent

Order(order_id="ORD-001", total=1499.00).discount        # None
Order(order_id="ORD-002", total=1499.00,
      discount={"code": "SAVE20", "percent_off": 20.0}).discount.code   # "SAVE20"